# Cube Accuracy vs Sequence Length

This notebook does a post-hoc length analysis for the `formal_langs_cube_*` eval variants. The eval artifacts include per-example `input_length` in the request metadata, so we can join requests with predictions and compute accuracy by length without rerunning any model evals.

By default, the notebook keeps only the final checkpoint for each plotted model class, where "final" means the maximum checkpoint step present locally for that model class among the selected cube evals.

In [ ]:
from __future__ import annotations

import json
import math
import re
import sys
from pathlib import Path

from IPython.display import display
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "aesthetics.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import aesthetics as aes  # noqa: E402

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_rows", 120)

## Config

In [ ]:
RESULTS_PATH = PROJECT_ROOT / "results" / "jacksonp-hybrid-small-downstream-evals" / "all_results.json"

CUBE_TASK_RE = re.compile(
    r"formal_langs_cube_(unique|reassign_const|reassign_var|undefined):(v[246])"
)

TASK_DISPLAY = {
    "unique": "Unique",
    "reassign const": "Reassign Const",
    "reassign var": "Reassign Var",
    "undefined": "Undefined",
}
CUBE_TASK_ORDER = ["unique", "reassign const", "reassign var", "undefined"]
VERSION_ORDER = ["v2", "v4", "v6"]

# Set to None to include every model class found in the local cube results.
MODEL_CLASSES_TO_PLOT = [
    "transformer 275M",
    "hybrid 275M",
    "transformer 1B",
    "hybrid 1B",
    # "GDN + 1B",
    # "GDN +/- 1B",
    "transformer 7B",
    "hybrid 7B",
]

# Input length is stored by the dataset as character count of the evaluated prompt.
LENGTH_BIN_WIDTH = 100
MIN_BIN_EXAMPLES = 10

SIZE_PAIR_ORDER = ["275M", "1B", "7B"]
POINT_DELTA_FORMATTER = mticker.FuncFormatter(lambda value, _: f"{value * 100:.0f} pts")

SIZE_PAIR_PALETTE = {
    "275M": "#8fd19e",
    "1B": "#2ca25f",
    "7B": "#006d2c",
}

MODEL_CLASS_PATTERNS = [
    (r"olmo3-hybrid-gdn-deux_1b", "hybrid 1B"),
    (r"olmo3-baseline-jacksonp_1b", "transformer 1B"),
    (r"transformer-small-cx100_275m", "transformer 275M"),
    (r"hybrid-small-cx100_275m", "hybrid 275M"),
    (r"pure-gdn-ladder-gate-neg-eig_1b", "GDN +/- 1B"),
    (r"pure-gdn-ladder-gate-pos-eig_1b", "GDN + 1B"),
    (r"allenai_olmo-3-1025-7b_stage1", "transformer 7B"),
    (r"allenai_olmo-hybrid-7b_stage1", "hybrid 7B"),
    (r"hybrid-aperiodic_supervised", "hybrid 275M + aperiodic supervised"),
    (r"hybrid-aperiodic_unsupervised", "hybrid 275M + aperiodic unsupervised"),
    (r"hybrid-periodic_supervised", "hybrid 275M + periodic supervised"),
    (r"hybrid-periodic_unsupervised", "hybrid 275M + periodic unsupervised"),
    (r"hybrid-r-trivial_supervised", "hybrid 275M + r-trivial supervised"),
    (r"hybrid-r-trivial_unsupervised", "hybrid 275M + r-trivial unsupervised"),
]

aes.update_model_palette(
    family="transformer",
    color="light:#DD0426",
    models=["7B", "1B", "275M"],
)
aes.update_model_palette(
    family="hybrid",
    color="light:#8304DE",
    models=["7B", "1B", "275M"],
)
aes.update_model_palette(
    family="GDN",
    color="light:#4b88fa",
    models=["+/- 1B", "+ 1B"],
)
MODEL_CLASS_PALETTE = aes.PALETTES["models"].copy()

## Helpers

In [ ]:
def extract_step(*values: str | None) -> int | None:
    for value in values:
        if not value:
            continue
        match = re.search(r"(?:^|[_/-])step(\d+)(?:\D|$)", value, flags=re.IGNORECASE)
        if match:
            return int(match.group(1))
    return None


def model_class_for(experiment_name: str, model: str) -> str:
    haystack = f"{experiment_name} {model}"
    for pattern, label in MODEL_CLASS_PATTERNS:
        if re.search(pattern, haystack, flags=re.IGNORECASE):
            return label

    parts = [part for part in Path(model.rstrip("/")).parts if part]
    if len(parts) >= 2 and re.match(r"step\d+", parts[-1], flags=re.IGNORECASE):
        return "/".join(parts[-3:-1]) if len(parts) >= 3 else parts[-2]
    return experiment_name.split("_step", 1)[0].split("-step", 1)[0]


def task_by_name(result: dict) -> dict[str, dict]:
    return {task["task"]: task for task in result.get("tasks", [])}


def artifact_file_for_task(result: dict, task_name: str, tasks: dict[str, dict], kind: str) -> Path | None:
    task = tasks.get(task_name)
    task_hash = task.get("task_hash") if task else None
    if not task_hash:
        return None

    suffix = "predictions" if kind == "predictions" else "requests"
    root = RESULTS_PATH.parent / result["experiment_id"] / "main" / kind
    matches = sorted(root.glob(f"*/*_{task_hash[-6:]}-{suffix}.jsonl"))
    return matches[0] if matches else None


def metric_value(instance_metrics: dict, metric: str | None) -> float | None:
    value = instance_metrics
    for key in (metric or "").split(":"):
        if not isinstance(value, dict) or key not in value:
            return None
        value = value[key]
    if isinstance(value, int | float) and pd.notna(value) and math.isfinite(value):
        return float(value)
    return None


def wilson_ci95(successes: pd.Series, n: pd.Series) -> tuple[pd.Series, pd.Series]:
    z = 1.96
    p = successes / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    radius = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denom
    return (center - radius).clip(lower=0), (center + radius).clip(upper=1)

def matched_hybrid_transformer_delta(df: pd.DataFrame, group_columns: list[str]) -> pd.DataFrame:
    pair_df = df.copy()
    pair_df["model_size"] = pair_df["model_class"].str.extract(r"(275M|1B|7B)")[0]
    pair_df["architecture"] = pair_df["model_class"].str.extract(r"^(hybrid|transformer)")[0]
    pair_df = pair_df[
        pair_df["model_size"].isin(SIZE_PAIR_ORDER)
        & pair_df["architecture"].isin(["hybrid", "transformer"])
    ].copy()

    value_columns = group_columns + ["model_size"]
    hybrid = pair_df[pair_df["architecture"] == "hybrid"].rename(
        columns={"accuracy": "hybrid_accuracy", "successes": "hybrid_successes", "n": "hybrid_n"}
    )
    transformer = pair_df[pair_df["architecture"] == "transformer"].rename(
        columns={
            "accuracy": "transformer_accuracy",
            "successes": "transformer_successes",
            "n": "transformer_n",
        }
    )

    delta_df = hybrid[value_columns + ["hybrid_accuracy", "hybrid_successes", "hybrid_n"]].merge(
        transformer[value_columns + ["transformer_accuracy", "transformer_successes", "transformer_n"]],
        on=value_columns,
        how="inner",
    )
    delta_df["delta_accuracy"] = delta_df["hybrid_accuracy"] - delta_df["transformer_accuracy"]
    delta_df["delta_se"] = np.sqrt(
        delta_df["hybrid_accuracy"] * (1 - delta_df["hybrid_accuracy"]) / delta_df["hybrid_n"]
        + delta_df["transformer_accuracy"]
        * (1 - delta_df["transformer_accuracy"])
        / delta_df["transformer_n"]
    )
    delta_df["ci_low"] = (delta_df["delta_accuracy"] - 1.96 * delta_df["delta_se"]).clip(lower=-1)
    delta_df["ci_high"] = (delta_df["delta_accuracy"] + 1.96 * delta_df["delta_se"]).clip(upper=1)
    delta_df["model_size"] = pd.Categorical(delta_df["model_size"], categories=SIZE_PAIR_ORDER, ordered=True)
    return delta_df.sort_values(value_columns)


## Select Final Checkpoints

In [ ]:
raw_results = json.loads(RESULTS_PATH.read_text())
result_by_id = {result["experiment_id"]: result for result in raw_results}

summary_rows = []
for result in raw_results:
    experiment_name = result.get("experiment_name", "")
    model = result.get("model", "")
    model_class = model_class_for(experiment_name, model)
    step = extract_step(experiment_name, model)
    tasks = task_by_name(result)

    for task_name, payload in result.get("summary", {}).items():
        match = CUBE_TASK_RE.fullmatch(task_name)
        if not match or task_name not in tasks:
            continue
        cube_task, version = match.groups()
        summary_rows.append(
            {
                "experiment_id": result.get("experiment_id"),
                "experiment_name": experiment_name,
                "model": model,
                "model_class": model_class,
                "step": step,
                "task": task_name,
                "cube_task": cube_task.replace("_", " "),
                "version": version,
                "metric": payload.get("metric"),
                "summary_score": payload.get("score"),
                "task_hash": tasks[task_name].get("task_hash"),
                "num_instances": tasks[task_name].get("num_instances"),
            }
        )

summary_df = pd.DataFrame(summary_rows)
summary_df["step"] = pd.to_numeric(summary_df["step"], errors="coerce")
summary_df["summary_score"] = pd.to_numeric(summary_df["summary_score"], errors="coerce")
summary_df = summary_df.dropna(subset=["step", "summary_score"]).copy()
summary_df["step"] = summary_df["step"].astype(int)

if MODEL_CLASSES_TO_PLOT is not None:
    summary_df = summary_df[summary_df["model_class"].isin(MODEL_CLASSES_TO_PLOT)].copy()

# If a rerun produced the same model class / checkpoint / task, keep the latest experiment id.
summary_df = (
    summary_df.sort_values(
        ["task", "model_class", "step", "experiment_id"],
        ascending=[True, True, True, False],
    )
    .drop_duplicates(["task", "model_class", "step"], keep="first")
    .sort_values(["model_class", "step", "task"])
)

final_steps = (
    summary_df.groupby("model_class", as_index=False)["step"]
    .max()
    .rename(columns={"step": "final_step"})
)
final_summary_df = summary_df.merge(final_steps, on="model_class")
final_summary_df = final_summary_df[final_summary_df["step"] == final_summary_df["final_step"]].copy()

coverage = (
    final_summary_df.groupby(["model_class", "final_step"], as_index=False)
    .agg(tasks=("task", "nunique"), examples=("num_instances", "sum"))
    .sort_values(["model_class", "final_step"])
)

print(f"Loaded {len(raw_results):,} experiments from {RESULTS_PATH}")
print(f"Selected {len(final_summary_df):,} final-checkpoint cube task rows")
display(coverage)

## Join Predictions With Length Metadata

In [ ]:
def item_rows_for_summary_row(row: pd.Series) -> list[dict]:
    result = result_by_id[row["experiment_id"]]
    tasks = task_by_name(result)
    prediction_path = artifact_file_for_task(result, row["task"], tasks, "predictions")
    request_path = artifact_file_for_task(result, row["task"], tasks, "requests")
    if prediction_path is None or request_path is None:
        raise FileNotFoundError(
            f"Missing artifacts for {row['experiment_id']} {row['task']}: "
            f"prediction_path={prediction_path}, request_path={request_path}"
        )

    item_rows = []
    with prediction_path.open() as prediction_file, request_path.open() as request_file:
        for prediction_line, request_line in zip(prediction_file, request_file, strict=True):
            prediction = json.loads(prediction_line)
            request = json.loads(request_line)
            doc = request.get("doc", {})

            if prediction.get("doc_id") != request.get("doc_id"):
                raise ValueError(
                    f"doc_id mismatch in {prediction_path.name}: "
                    f"prediction={prediction.get('doc_id')} request={request.get('doc_id')}"
                )

            score = metric_value(prediction.get("instance_metrics", {}), row["metric"])
            if score is None:
                continue

            item_rows.append(
                {
                    "experiment_id": row["experiment_id"],
                    "model_class": row["model_class"],
                    "step": row["step"],
                    "task": row["task"],
                    "cube_task": row["cube_task"],
                    "version": row["version"],
                    "doc_id": prediction.get("doc_id"),
                    "native_id": prediction.get("native_id"),
                    "input_length": doc.get("input_length"),
                    "num_variables": doc.get("num_variables"),
                    "num_assignments": doc.get("num_assignments"),
                    "binding_language": doc.get("binding_language"),
                    "score": score,
                }
            )
    return item_rows


item_rows = []
for _, row in final_summary_df.iterrows():
    item_rows.extend(item_rows_for_summary_row(row))

item_df = pd.DataFrame(item_rows)
for column in ["input_length", "num_variables", "num_assignments", "score"]:
    item_df[column] = pd.to_numeric(item_df[column], errors="coerce")
item_df = item_df.dropna(subset=["input_length", "score"]).copy()
item_df["cube_task"] = pd.Categorical(item_df["cube_task"], categories=CUBE_TASK_ORDER, ordered=True)
item_df["version"] = pd.Categorical(item_df["version"], categories=VERSION_ORDER, ordered=True)

print(f"Joined {len(item_df):,} final-checkpoint examples with length metadata")
display(
    item_df.groupby(["model_class", "cube_task", "version"], observed=True)
    .agg(examples=("score", "size"), accuracy=("score", "mean"), min_length=("input_length", "min"), max_length=("input_length", "max"))
    .reset_index()
    .head(24)
)

## Accuracy vs Input Length

The x-axis uses `input_length`, which is the dataset-provided character length of the evaluated prompt. Points are 100-character bins by default; bands show Wilson 95% intervals for binomial accuracy.

In [ ]:
length_min = math.floor(item_df["input_length"].min() / LENGTH_BIN_WIDTH) * LENGTH_BIN_WIDTH
length_max = math.ceil(item_df["input_length"].max() / LENGTH_BIN_WIDTH) * LENGTH_BIN_WIDTH
bins = np.arange(length_min, length_max + LENGTH_BIN_WIDTH, LENGTH_BIN_WIDTH)

item_df["length_bin"] = pd.cut(
    item_df["input_length"],
    bins=bins,
    include_lowest=True,
    right=False,
)
item_df["length_bin_mid"] = item_df["length_bin"].map(lambda interval: float(interval.mid))

length_df = (
    item_df.groupby(
        ["cube_task", "version", "model_class", "length_bin", "length_bin_mid"],
        observed=True,
        as_index=False,
    )
    .agg(accuracy=("score", "mean"), successes=("score", "sum"), n=("score", "size"))
)
length_df = length_df[length_df["n"] >= MIN_BIN_EXAMPLES].copy()
length_df["ci_low"], length_df["ci_high"] = wilson_ci95(length_df["successes"], length_df["n"])

hue_order = [
    model_class
    for model_class in (MODEL_CLASSES_TO_PLOT or sorted(length_df["model_class"].unique()))
    if model_class in set(length_df["model_class"])
]
missing_palette = dict(zip(hue_order, sns.color_palette("tab10", n_colors=len(hue_order)), strict=False))
palette = {model_class: MODEL_CLASS_PALETTE.get(model_class, missing_palette[model_class]) for model_class in hue_order}

n_rows = len([task for task in CUBE_TASK_ORDER if task in set(length_df["cube_task"].dropna())])
n_cols = len([version for version in VERSION_ORDER if version in set(length_df["version"].dropna())])
fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(aes.PAPER_WIDTH_IN * 1.55, 6.6),
    sharex=True,
    sharey=True,
    squeeze=False,
)

for row_idx, cube_task in enumerate(CUBE_TASK_ORDER):
    for col_idx, version in enumerate(VERSION_ORDER):
        ax = axes[row_idx, col_idx]
        panel_df = length_df[(length_df["cube_task"] == cube_task) & (length_df["version"] == version)]

        for model_class in hue_order:
            model_df = panel_df[panel_df["model_class"] == model_class].sort_values("length_bin_mid")
            if model_df.empty:
                continue
            color = palette[model_class]
            x = model_df["length_bin_mid"].astype(float).to_numpy()
            y = model_df["accuracy"].astype(float).to_numpy()
            low = model_df["ci_low"].astype(float).to_numpy()
            high = model_df["ci_high"].astype(float).to_numpy()
            ax.plot(x, y, marker="o", markersize=2.4, linewidth=1.1, color=color, label=model_class)
            ax.fill_between(x, low, high, color=color, alpha=0.10, linewidth=0)

        ax.axhline(0.25, color=aes.NONSEMANTIC_COLOR, linestyle="--", linewidth=0.9, alpha=0.75)
        ax.grid(True, alpha=0.25)
        ax.set_ylim(0, 1.02)
        ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
        ax.xaxis.set_major_formatter(aes.COMPACT_NUMBER_FORMATTER)

        if row_idx == 0:
            ax.set_title(version, fontweight="regular", size=9)
        if col_idx == 0:
            ax.set_ylabel(TASK_DISPLAY[cube_task])
        else:
            ax.tick_params(labelleft=False)
        if row_idx == n_rows - 1:
            ax.set_xlabel("Input length (characters)")
        else:
            ax.tick_params(labelbottom=False)

handles = [
    plt.Line2D([0], [0], color=palette[model_class], marker="o", linewidth=1.3, markersize=3, label=model_class)
    for model_class in hue_order
]
fig.legend(
    handles=handles,
    loc="center left",
    bbox_to_anchor=(1.0, 0.5),
    frameon=False,
    fontsize="small",
)
fig.suptitle("Cube Eval Accuracy vs Input Length, Final Checkpoints Only", y=1.01, fontsize=11)
fig.tight_layout()
plt.show()

## Matched Hybrid Minus Transformer Differences

These panels use the same length bins as above, but plot `hybrid accuracy - transformer accuracy` after matching model size. The GDN models are excluded. Values above zero favor the hybrid model for that size class.

In [ ]:
length_delta_df = matched_hybrid_transformer_delta(
    length_df,
    group_columns=["cube_task", "version", "length_bin", "length_bin_mid"],
)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(aes.PAPER_WIDTH_IN * 1.55, 6.6),
    sharex=True,
    sharey=True,
    squeeze=False,
)

for row_idx, cube_task in enumerate(CUBE_TASK_ORDER):
    for col_idx, version in enumerate(VERSION_ORDER):
        ax = axes[row_idx, col_idx]
        panel_df = length_delta_df[
            (length_delta_df["cube_task"] == cube_task) & (length_delta_df["version"] == version)
        ]

        for model_size in SIZE_PAIR_ORDER:
            size_df = panel_df[panel_df["model_size"] == model_size].sort_values("length_bin_mid")
            if size_df.empty:
                continue
            color = SIZE_PAIR_PALETTE[model_size]
            x = size_df["length_bin_mid"].astype(float).to_numpy()
            y = size_df["delta_accuracy"].astype(float).to_numpy()
            low = size_df["ci_low"].astype(float).to_numpy()
            high = size_df["ci_high"].astype(float).to_numpy()
            ax.plot(x, y, marker="o", markersize=2.4, linewidth=1.1, color=color, label=model_size)
            ax.fill_between(x, low, high, color=color, alpha=0.12, linewidth=0)

        ax.axhline(0, color=aes.NONSEMANTIC_COLOR, linestyle="-", linewidth=0.9, alpha=0.9)
        ax.grid(True, alpha=0.25)
        ax.set_ylim(-0.55, 0.55)
        ax.yaxis.set_major_formatter(POINT_DELTA_FORMATTER)
        ax.xaxis.set_major_formatter(aes.COMPACT_NUMBER_FORMATTER)

        if row_idx == 0:
            ax.set_title(version, fontweight="regular", size=9)
        if col_idx == 0:
            ax.set_ylabel(TASK_DISPLAY[cube_task])
        else:
            ax.tick_params(labelleft=False)
        if row_idx == n_rows - 1:
            ax.set_xlabel("Input length (characters)")
        else:
            ax.tick_params(labelbottom=False)

size_handles = [
    plt.Line2D([0], [0], color=SIZE_PAIR_PALETTE[size], marker="o", linewidth=1.3, markersize=3, label=size)
    for size in SIZE_PAIR_ORDER
    if size in set(length_delta_df["model_size"].dropna())
]
fig.legend(
    handles=size_handles,
    loc="center left",
    bbox_to_anchor=(1.0, 0.5),
    frameon=False,
    fontsize="small",
    title="Size",
    title_fontsize="small",
)
fig.suptitle("Cube Eval Hybrid - Transformer Accuracy vs Input Length", y=1.01, fontsize=11)
fig.tight_layout()
plt.show()

## Optional: Collapsed Across Cube Versions

This view keeps the same final-checkpoint filter but averages across `v2`, `v4`, and `v6` within each cube task. It is useful when you want the broad length trend without twelve panels.

In [ ]:
collapsed_df = (
    item_df.groupby(["cube_task", "model_class", "length_bin", "length_bin_mid"], observed=True, as_index=False)
    .agg(accuracy=("score", "mean"), successes=("score", "sum"), n=("score", "size"))
)
collapsed_df = collapsed_df[collapsed_df["n"] >= MIN_BIN_EXAMPLES].copy()
collapsed_df["ci_low"], collapsed_df["ci_high"] = wilson_ci95(collapsed_df["successes"], collapsed_df["n"])

fig, axes = plt.subplots(
    1,
    len(CUBE_TASK_ORDER),
    figsize=(aes.PAPER_WIDTH_IN * 1.6, 1.9),
    sharex=True,
    sharey=True,
    squeeze=False,
)

for col_idx, cube_task in enumerate(CUBE_TASK_ORDER):
    ax = axes[0, col_idx]
    panel_df = collapsed_df[collapsed_df["cube_task"] == cube_task]
    for model_class in hue_order:
        model_df = panel_df[panel_df["model_class"] == model_class].sort_values("length_bin_mid")
        if model_df.empty:
            continue
        color = palette[model_class]
        x = model_df["length_bin_mid"].astype(float).to_numpy()
        y = model_df["accuracy"].astype(float).to_numpy()
        ax.plot(x, y, marker="o", markersize=2.4, linewidth=1.1, color=color, label=model_class)
    ax.axhline(0.25, color=aes.NONSEMANTIC_COLOR, linestyle="--", linewidth=0.9, alpha=0.75)
    ax.set_title(TASK_DISPLAY[cube_task], fontweight="regular", size=9)
    ax.grid(True, alpha=0.25)
    ax.set_ylim(0, 1.02)
    ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
    ax.xaxis.set_major_formatter(aes.COMPACT_NUMBER_FORMATTER)
    ax.set_xlabel("Input length")
    if col_idx == 0:
        ax.set_ylabel("Accuracy")
    else:
        ax.tick_params(labelleft=False)

fig.legend(handles=handles, loc="center left", bbox_to_anchor=(1.0, 0.5), frameon=False, fontsize="small")
fig.suptitle("Cube Eval Accuracy vs Input Length, Versions Collapsed", y=1.05, fontsize=11)
fig.tight_layout()
plt.show()

## Optional: Collapsed Matched Differences

This version collapses across `v2`, `v4`, and `v6` before taking matched-size hybrid-minus-transformer differences.

In [ ]:
collapsed_delta_df = matched_hybrid_transformer_delta(
    collapsed_df,
    group_columns=["cube_task", "length_bin", "length_bin_mid"],
)

fig, axes = plt.subplots(
    1,
    len(CUBE_TASK_ORDER),
    figsize=(aes.PAPER_WIDTH_IN * 1.6, 1.9),
    sharex=True,
    sharey=True,
    squeeze=False,
)

for col_idx, cube_task in enumerate(CUBE_TASK_ORDER):
    ax = axes[0, col_idx]
    panel_df = collapsed_delta_df[collapsed_delta_df["cube_task"] == cube_task]
    for model_size in SIZE_PAIR_ORDER:
        size_df = panel_df[panel_df["model_size"] == model_size].sort_values("length_bin_mid")
        if size_df.empty:
            continue
        color = SIZE_PAIR_PALETTE[model_size]
        x = size_df["length_bin_mid"].astype(float).to_numpy()
        y = size_df["delta_accuracy"].astype(float).to_numpy()
        ax.plot(x, y, marker="o", markersize=2.4, linewidth=1.1, color=color, label=model_size)
    ax.axhline(0, color=aes.NONSEMANTIC_COLOR, linestyle="-", linewidth=0.9, alpha=0.9)
    ax.set_title(TASK_DISPLAY[cube_task], fontweight="regular", size=9)
    ax.grid(True, alpha=0.25)
    ax.set_ylim(-0.55, 0.55)
    ax.yaxis.set_major_formatter(POINT_DELTA_FORMATTER)
    ax.xaxis.set_major_formatter(aes.COMPACT_NUMBER_FORMATTER)
    ax.set_xlabel("Input length")
    if col_idx == 0:
        ax.set_ylabel("Hybrid - transformer")
    else:
        ax.tick_params(labelleft=False)

fig.legend(
    handles=size_handles,
    loc="center left",
    bbox_to_anchor=(1.0, 0.5),
    frameon=False,
    fontsize="small",
    title="Size",
    title_fontsize="small",
)
fig.suptitle("Cube Eval Hybrid - Transformer Accuracy vs Input Length, Versions Collapsed", y=1.05, fontsize=11)
fig.tight_layout()
plt.show()